<center>
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/1/1d/Logo_T%C3%A9l%C3%A9com_SudParis.svg/1014px-Logo_T%C3%A9l%C3%A9com_SudParis.svg.png" width="10%" />
</center>

<center> <h2> NET 4103/7431 Complex Network </h2> </center>

<center> <h3> Vincent Gauthier (vincent.gauthier@telecom-sudparis.eu) </h3> </center>

### Note
Avant de commencer les exercices, assurez-vous que tout fonctionne comme prévu. Tout d'abord, le redémarrage du kernel **(dans la barre de menus, sélectionnez le kernel $\rightarrow$ Restart)**.

Assurez-vous que vous remplir les célluler aux endroits marquer «YOUR CODE HERE». 

Veuillez supprimer les ligne «raise NotImplementedError()» dans toutes les cellules auxquelles vous avez répondu, ainsi que votre nom et prénom ci-dessous:

In [ ]:
NOM = "XXX"
PRENOM = "XXX"

---

## Dependencies

This section explains the purpose of the following code; each method documents its parameters, return value, and expected types.


In [ ]:
import duckdb
import spacy
from tqdm.auto import tqdm
import pandas as pd
import requests

from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
from itertools import permutations

## Optional spaCy model setup

This section explains the purpose of the following code; each method documents its parameters, return value, and expected types.


In [ ]:
#!python -m spacy download en_core_web_lg

## Wikidata API client and method documentation

This section explains the purpose of the following code; each method documents its parameters, return value, and expected types.


In [ ]:
from typing import Optional
class WikiDataKG:
    """
    Wrapper for Wikidata APIs to link entities and discover relationships.

    possible endpoint : 
    - "https://query.wikidata.org/sparql"
    - "https://qlever.cs.uni-freiburg.de/api/wikidata"
    """
    
    def __init__(self, endpoint: str = "https://query.wikidata.org/sparql") -> None:
        """
        Initialize the WikiDataKG instance with a User-Agent header.
        The User-Agent is required by Wikidata's API to identify the client.
        """
        self.endpoint = endpoint
        self.session = requests.Session()
        self.session.headers["User-Agent"] = "KGRagDemo/1.0 (vgauthier@luxbulb.org)"

    def _sparql(self, query: str) -> list[dict[str, dict[str, str]]]:
        """Run a SPARQL query and return its result bindings.

        Args:
            query: SPARQL query text.

        Returns:
            Result binding dictionaries from the endpoint.
        """
        r = self.session.get(self.endpoint,
                        params={"query": query, "format": "json"}, timeout=30)
        r.raise_for_status()
        return r.json()["results"]["bindings"]
        
    def relations_batch(self, qids: list[str], chunk: int = 50) -> list[tuple[str, str, str]]:
        """Retrieve selected outgoing relations for Wikidata entities.

        Args:
            qids: Wikidata entity IDs to query.
            chunk: Maximum number of IDs per SPARQL request.

        Returns:
            Tuples of (subject Q-ID, relation label, value label).
        """
        props = "wdt:P31 wdt:P106 wdt:P800 wdt:P27 wdt:P19 wdt:P4970"
        out = []
        progress_bar = tqdm(total=len(qids), desc=f"Processing Relations")
        for i in range(0, len(qids), chunk):
            values = " ".join(f"wd:{q}" for q in qids[i:i + chunk])
            query = f"""
            SELECT ?item ?propLabel ?valueLabel WHERE {{
              VALUES ?item {{ {values} }}
              VALUES ?p {{ {props} }}
              ?item ?p ?value .
              ?prop wikibase:directClaim ?p .
              SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
            }}
            """
            try:
                rows = self._sparql(query)
            except requests.exceptions.RequestException as e:
                print(f"Relations batch failed: {e}")
                continue
            out.extend((r["item"]["value"].rsplit("/", 1)[-1],
                        r["propLabel"]["value"],
                        r["valueLabel"]["value"]) for r in rows)
            progress_bar.update(chunk)
        return out
    
    def _canon(self, n: str) -> str:
        """Normalize a name for matching an English Wikipedia title.

        Args:
            n: Entity name to normalize.
        
        Returns:
            The stripped name with underscores replaced and initial capitalization.
        """
        n = n.strip().replace("_", " ")
        return n[:1].upper() + n[1:]

    def _esc(self, s: str) -> str:
        """Escape a value for inclusion in a SPARQL string literal.

        Args:
            s: Text to escape.
        
        Returns:
            Text with backslashes and double quotes escaped.
        """
        return s.replace("\\", "\\\\").replace('"', '\\"')

    def link_batch(self, names: list[str], chunk: int = 50) -> dict[str, tuple[str, str]]:
        """Resolve multiple entity names through Wikipedia sitelinks.

        Args:
            names: Entity names to resolve.
            chunk: Maximum number of names per query.
        
        Returns:
            Mapping from matched input names to (Q-ID, description) pairs.
        """
        progress_bar = tqdm(total=len(names), desc=f"Processing Links")
        out = {}
        for i in range(0, len(names), chunk):
            part = names[i:i + chunk]
            values = " ".join(f'("{self._esc(n)}" "{self._esc(self._canon(n))}"@en)' for n in part)
            query = f"""
            SELECT ?name ?item ?desc 
            WHERE {{
              VALUES (?name ?title) {{ {values} }}
              ?article schema:about ?item ;
                       schema:isPartOf <https://en.wikipedia.org/> ;
                       schema:name ?title .
              FILTER NOT EXISTS {{ ?item wdt:P31 wd:Q4167410 }}   # skip disambiguation pages
              OPTIONAL {{ ?item schema:description ?desc . FILTER(LANG(?desc) = "en") }}
            }}"""
            try:
                rows = self._sparql(query)
            except requests.exceptions.RequestException as e:
                print(f"link batch failed: {e}")
                continue
            except requests.exceptions.HTTPError as e:
                print(f"link batch failed: {e}")
                continue
            for r in rows:
                out[r["name"]["value"]] = (r["item"]["value"].rsplit("/", 1)[-1],
                                           r.get("desc", {}).get("value", ""))
            progress_bar.update(chunk)
        return out

## Relation lookup example

This section explains the purpose of the following code; each method documents its parameters, return value, and expected types.


In [ ]:
kg = WikiDataKG()
kg.relations_batch(["Q3052772", "Q90"])

## DuckDB entity-storage pipeline

This section explains the purpose of the following code; each method documents its parameters, return value, and expected types.


In [ ]:
class DBStore:
    def __init__(self, db_path: str = "./database-p4.db", embedding_size:int = 384) -> None:
        """
        Initialize 

        Args:
            db_path: Path to database file
        """
        self.conn = duckdb.connect(db_path)
        self.embedding_size = embedding_size
        self.model = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dimensions
        self.kg = WikiDataKG()
        # Load spaCy for NLP
        try:
            self.nlp = spacy.load('en_core_web_lg')
            print("Loaded spaCy NLP model")
        except OSError:
            print("Warning: spaCy model not found. Install with: python -m spacy download en_core_web_sm")
            self.nlp = None

        print("Pipeline initialization complete\n")

    def _create_entities_table(self) -> None:
        """Create the document, linked-entity, and relation tables."""
        try: 
            self.conn.execute(f"""
            DROP TABLE IF EXISTS entities;
            CREATE TABLE IF NOT EXISTS doc 
            (
                doc_id UInt32,
                text String,
            );

            DROP TABLE IF EXISTS linked_entities;
            CREATE TABLE IF NOT EXISTS linked_entities 
            (
                doc_id UInt32, 
                text String, 
                label String, 
                wikidata_id String,
                description String
            );

            DROP TABLE IF EXISTS entity_relations;
            CREATE TABLE IF NOT EXISTS entity_relations (
                subject_qid String, 
                relation String,
                value String
            );

            DROP TABLE IF EXISTS embedding;
            CREATE TABLE IF NOT EXISTS embedding (
                doc_id UInt32, 
                text String,
                context String,
                embedding FLOAT[{self.embedding_size}]
            );
            """)
        except Exception as e:
            print(f"Warning when creating entities: {str(e)}")

    def generate_embedding(self):
        """
        Retrive main documents and their related context and generetae embedding
        """
        # Retrive main doc + context
        df_context = self.conn.sql("""
        WITH documents AS (
            SELECT * FROM doc
        ), 
        context AS (
            SELECT
                le.doc_id AS doc_id,
                concat_ws(
                    '; ',
                    string_agg(DISTINCT le.description, '; '),
                    string_agg(DISTINCT en.value, '; ')
                ) AS context
                FROM linked_entities AS le
                JOIN entity_relations AS en
                    ON le.wikidata_id = en.subject_qid
                GROUP BY le.doc_id
        ) 
        SELECT 
            d.doc_id,
            d.text,
            c.context
        FROM documents d
        LEFT JOIN context c ON d.doc_id = c.doc_id
        ORDER BY doc_id
        """).df()
        # generate main text embedding
        embeddings_text = self.model.encode(
            df_context["text"].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,   # useful for cosine similarity
        )
        # generate main text embedding
        embeddings_context = self.model.encode(
            df_context["context"].fillna("").tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,   # useful for cosine similarity
        )
        emb = 0.7 * embeddings_text + 0.3 * embeddings_context
        df_context["embedding"] = emb.tolist()
        self.conn.execute("""
        INSERT INTO embedding 
        SELECT 
            doc_id,
            text,
            context,
            embedding
        FROM df_context
        """)

    def add_documents(self, documents:list[str]) -> None:
        """Extract entities, link them, fetch relations, and persist the results.

        Args:
            documents: Passage texts to process.
        
        Returns:
            None. Rows are persisted in DuckDB.
        """
        self._create_entities_table()
        wanted = {"GPE", "LOC", "ORG", "PERSON"}
        doc_rows, mentions = [], []
        # Define the progress bar using the estimated_total
        progress_bar = tqdm(total=len(documents), desc=f"Processing NER")
        # Pass 1: Extract NER
        for doc_id, spacy_doc in enumerate(self.nlp.pipe(documents, batch_size=64)):
            doc_rows.append((doc_id, documents[doc_id]))
            seen = set()
            # Pass 1: NER only, no network
            for ent in spacy_doc.ents:
                if ent.label_ in wanted and ent.text not in seen:
                    seen.add(ent.text)
                    mentions.append((doc_id, ent.text, ent.label_))
            progress_bar.update(1)
        
        # Pass 2: link every unique name once
        links = self.kg.link_batch([m[1] for m in mentions])
        ent_rows = [(d, t, l, links[t][0], links[t][1])
                    for d, t, l in mentions if links.get(t)]

        # Pass 3: relations, once per unique QID
        qids = sorted({r[3] for r in ent_rows})
        rel_rows = self.kg.relations_batch(qids)
        df_doc = pd.DataFrame(doc_rows, columns=["doc_id", "text"])
        self.conn.execute("INSERT INTO doc SELECT * FROM df_doc")
        df_ent = pd.DataFrame(ent_rows, columns=["doc_id","text","label","wikidata_id","description"])
        self.conn.execute("INSERT INTO linked_entities SELECT * FROM df_ent")
        self.conn.executemany("INSERT INTO entity_relations VALUES (?, ?, ?)", rel_rows)

    def search(self, query: str, topk: int = 5) -> pd.DataFrame:
        try:
            query_embedding = self.model.encode([query], normalize_embeddings=True).tolist()[0]
    
            db_query = f"""
            SELECT 
                doc_id, 
                text,
                context,
                array_distance(embedding, {query_embedding}::FLOAT[{self.embedding_size}]) AS distance,
                RANK() OVER (ORDER BY distance) AS rank
            FROM embedding
            ORDER BY array_distance(embedding, {query_embedding}::FLOAT[{self.embedding_size}])
            LIMIT {topk};
            """
            return self.conn.sql(db_query).df()
        except Exception as e:
            print(f"Warning when doing query: {str(e)}")
    
    def close(self) -> None:
        """Close the DuckDB connection."""
        self.conn.close()

## Dataset loading and pipeline execution

This section explains the purpose of the following code; each method documents its parameters, return value, and expected types.


In [ ]:
corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus")["passages"]
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer")["test"]

db = DBStore()
db._create_entities_table()
db.add_documents(corpus["passage"])
db.generate_embedding()
db.close()

In [ ]:
db = DBStore()
print(qa["question"][0])
print("================")
response = db.search(qa["question"][0])
print(response)

print("================")
print(qa["question"][758])
response= db.search(qa["question"][758])
print(response)

print("================")
print(qa["question"][32])
response= db.search(qa["question"][32])
print(response)

print("================")
print(qa["question"][133])
response= db.search(qa["question"][133])
print(response)

print("================")
print(qa["question"][129])
response= db.search(qa["question"][129])
print(response)


In [ ]:
def build_prompt(df: pd.DataFrame, question: str) -> str:
    text = df.loc[0, "text"]
    context = df.loc[0, "context"]
    prompt = f"""
    Answer the question using only the context below.
    If the answer is not in the context, say you don't know.
        
    Context 1:
    {context}
    
    Context 2:
    {text}
    
    Question: {question}
    Answer:
    """
    return prompt

In [ ]:
print("================")
print(qa["question"][32])
response= db.search(qa["question"][32])
print(response)

print(build_prompt(response, qa["question"][32]))

In [ ]:
db.close()